In [1]:
import os
import json
from pathlib import Path
from typing import Any, Dict, List, TypedDict

from dotenv import load_dotenv

# --- SET YOUR PROJECT CONFIG HERE (Overrides .env) ---
os.environ["VERTEXAI_PROJECT"] = ""
os.environ["VERTEXAI_LOCATION"] = "us-central1"
os.environ["LLM_BACKEND"] = "vertex_ai"
os.environ["MODEL_NAME"] = "gemini-2.5-flash"
os.environ["LANGCHAIN_TRACING_V2"] = "false"

from langgraph.graph import END, StateGraph

# Import your local custom tools
from tools.docx_parser_tool import parse_docx
from tools.protocol_table_extractor import extract_tables
from tools.template_signal_detector import detect_update_units
from config.llm_config_langchain import get_llm

# Setup base directory (Jupyter-friendly)
base_dir = Path(os.getcwd())

# Load .env from the current project directory as fallback
load_dotenv(base_dir / ".env")

# Optional debug prints
print("--- Environment Variables ---")
print(f"LLM_BACKEND={os.getenv('LLM_BACKEND')}")
print(f"MODEL_NAME={os.getenv('MODEL_NAME')}")
print(f"VERTEXAI_PROJECT={os.getenv('VERTEXAI_PROJECT')}")

/home/jupyter/phds/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.cloud.aiplatform_v1beta1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.aiplatform_v1beta1 past that date.
  warnings.warn(message, FutureWarning)
/home/jupyter/phds/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.cloud.aiplatform_v1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.aiplatform_v1 past that date.
  warnings.warn(message, FutureWarning)
/home/jupyter/phds/lib/python3.10/site-packages/google/api_c

--- Environment Variables ---
LLM_BACKEND=vertex_ai
MODEL_NAME=gemini-2.5-flash
VERTEXAI_PROJECT=


/home/jupyter/phds/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.cloud.vectorsearch_v1beta once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.vectorsearch_v1beta past that date.
  warnings.warn(message, FutureWarning)


In [2]:
class DemoState(TypedDict, total=False):
    # Input paths
    protocol_path: str
    template_path: str
    protocol_json_path: str
    template_json_path: str
    protocol_tables_json_path: str
    update_units_json_path: str
    summary_txt_path: str

    # Intermediate / output objects
    parsed_protocol: Dict[str, Any]
    parsed_template: Dict[str, Any]
    protocol_tables: List[Dict[str, Any]]
    update_units: List[Dict[str, Any]]
    summary_text: str

In [3]:
def _write_json(path: str, data) -> None:
    """Write JSON data to disk with UTF-8 encoding."""
    output_path = Path(path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(
        json.dumps(data, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

def _write_text(path: str, text: str) -> None:
    """Write plain text to disk."""
    output_path = Path(path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(text, encoding="utf-8")

def parse_protocol_node(state: DemoState) -> DemoState:
    """Parse protocol.docx and save structured JSON."""
    parsed_protocol = parse_docx(state["protocol_path"])
    _write_json(state["protocol_json_path"], parsed_protocol)
    return {"parsed_protocol": parsed_protocol}

def parse_template_node(state: DemoState) -> DemoState:
    """Parse sap_template.docx and save structured JSON."""
    parsed_template = parse_docx(state["template_path"])
    _write_json(state["template_json_path"], parsed_template)
    return {"parsed_template": parsed_template}

def extract_tables_node(state: DemoState) -> DemoState:
    """Extract protocol tables from parsed protocol JSON."""
    protocol_tables = extract_tables(state["parsed_protocol"])
    _write_json(state["protocol_tables_json_path"], protocol_tables)
    return {"protocol_tables": protocol_tables}

def detect_signals_node(state: DemoState) -> DemoState:
    """Detect update units from the parsed SAP template."""
    update_units = detect_update_units(state["parsed_template"])
    _write_json(state["update_units_json_path"], update_units)
    return {"update_units": update_units}

def summarize_outputs_node(state: DemoState) -> DemoState:
    """
    Use the LLM only at the final step to summarize what the workflow produced.
    """
    llm = get_llm()

    protocol_para_count = state["parsed_protocol"]["paragraph_count"]
    protocol_table_count = state["parsed_protocol"]["table_count"]
    template_para_count = state["parsed_template"]["paragraph_count"]
    template_table_count = state["parsed_template"]["table_count"]
    extracted_table_count = len(state["protocol_tables"])
    detected_unit_count = len(state["update_units"])

    prompt = f"""
You are helping explain a LangGraph demo for a short course on document automation in clinical trial documents.

Write a concise, clear summary in plain English for students.
Keep it short but informative. Use bullet points.

Facts:
- Protocol paragraphs: {protocol_para_count}
- Protocol tables found during parsing: {protocol_table_count}
- Protocol tables extracted into output artifact: {extracted_table_count}
- SAP template paragraphs: {template_para_count}
- SAP template tables: {template_table_count}
- Detected update units: {detected_unit_count}

Please explain:
1. What the workflow did
2. What files/artifacts were produced
3. Why this LangGraph version is more structured than pure Python
4. Why this version is more controlled than a CrewAI agent workflow
""".strip()

    response = llm.invoke(prompt)

    if hasattr(response, "content"):
        # Fix for Gemini models via VertexAI returning lists
        if isinstance(response.content, list):
            summary_text = "\n".join([
                block.get("text", "") if isinstance(block, dict) else str(block)
                for block in response.content
            ])
        else:
            summary_text = response.content
    else:
        summary_text = str(response)

    _write_text(state["summary_txt_path"], summary_text)
    return {"summary_text": summary_text}

In [4]:
def build_graph():
    """
    Build a minimal teaching-friendly LangGraph workflow.
    Flow: parse_protocol -> parse_template -> extract_tables -> detect_signals -> summarize_outputs -> END
    """
    graph = StateGraph(DemoState)

    # 1. Add nodes
    graph.add_node("parse_protocol", parse_protocol_node)
    graph.add_node("parse_template", parse_template_node)
    graph.add_node("extract_tables", extract_tables_node)
    graph.add_node("detect_signals", detect_signals_node)
    graph.add_node("summarize_outputs", summarize_outputs_node)

    # 2. Add edges
    graph.set_entry_point("parse_protocol")
    graph.add_edge("parse_protocol", "parse_template")
    graph.add_edge("parse_template", "extract_tables")
    graph.add_edge("extract_tables", "detect_signals")
    graph.add_edge("detect_signals", "summarize_outputs")
    graph.add_edge("summarize_outputs", END)

    return graph.compile()

In [5]:
print("--- Initializing LangGraph Workflow ---")

data_dir = base_dir / "data"
output_dir = base_dir / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

# Define the starting payload
initial_state = {
    "protocol_path": str(data_dir / "protocol.docx"),
    "template_path": str(data_dir / "sap_template.docx"),
    "protocol_json_path": str(output_dir / "protocol_parsed.json"),
    "template_json_path": str(output_dir / "sap_template_parsed.json"),
    "protocol_tables_json_path": str(output_dir / "protocol_tables.json"),
    "update_units_json_path": str(output_dir / "update_units.json"),
    "summary_txt_path": str(output_dir / "summary.txt"),
}

# Compile and run
app = build_graph()

run_config = {
    "recursion_limit": 50,
    "max_concurrency": 2
}

# Pass the config dictionary
final_state = app.invoke(initial_state, config=run_config)

# Generate and save a deterministic summary payload
summary = {
    "protocol_paragraphs": final_state["parsed_protocol"]["paragraph_count"],
    "protocol_tables_from_parser": final_state["parsed_protocol"]["table_count"],
    "protocol_tables_extracted": len(final_state["protocol_tables"]),
    "sap_template_paragraphs": final_state["parsed_template"]["paragraph_count"],
    "sap_template_tables": final_state["parsed_template"]["table_count"],
    "detected_update_units": len(final_state["update_units"]),
    "summary_txt_path": final_state["summary_txt_path"],
}

summary_path = output_dir / "demo_summary.json"
summary_path.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("\n--- LangGraph Demo Completed Successfully ---")
print(json.dumps(summary, indent=2, ensure_ascii=False))

--- Initializing LangGraph Workflow ---


/home/jupyter/phdemo/demo3_ipynb/config/llm_config_langchain.py:40: LangChainDeprecationWarning: The class `ChatVertexAI` was deprecated in LangChain 3.2.0 and will be removed in 4.0.0. An updated version of the class exists in the `langchain-google-genai package and should be used instead. To use it run `pip install -U `langchain-google-genai` and import as `from `langchain_google_genai import ChatGoogleGenerativeAI``.
  base_llm = ChatVertexAI(



--- LangGraph Demo Completed Successfully ---
{
  "protocol_paragraphs": 218,
  "protocol_tables_from_parser": 1,
  "protocol_tables_extracted": 1,
  "sap_template_paragraphs": 109,
  "sap_template_tables": 1,
  "detected_update_units": 19,
  "summary_txt_path": "/home/jupyter/phdemo/demo3_ipynb/outputs/summary.txt"
}
